# Kakao Pay PoC — Step 1: S3 → Delta 적재 (Auto Loader)

## 데이터 흐름

```
카카오페이 온프렘  →  AWS S3 업로드 (카카오페이 담당)  →  Databricks 적재 (지민 담당)
```

## Workflow 구조

```
step1_tiara  (group=tiara) → 클러스터 A  → SQS: kpay-poc-tiara-queue
step1_batch  (group=batch) → 클러스터 B  → SQS: kpay-poc-batch-queue
step1_cdc    (group=cdc)   → 클러스터 C  → SQS: kpay-poc-cdc-queue
        ↓ 전부 완료
    step2.ipynb → 클러스터 D  (SDP + Expectation)
```

> 노트북 파일 1개, group 파라미터로 분기 — DataFrame 스키마·처리 로직이 그룹마다 다름

## 그룹별 특성

| group | 데이터 | 특성 | trigger |
|:---|:---|:---|:---|
| **tiara** | 티아라 행동로그 | 75컬럼, 지속 스트리밍, 230GB/day | processingTime |
| **batch** | 결제·Kudu·ETL | 270컬럼(결제), 1회성, 36TB | availableNow |
| **cdc** | MySQL CDC (DMS) | Op=I/U/D 포함, APPLY CHANGES INTO | processingTime |

## SQS 큐 구조

```
S3 버킷
  ├── poc/tiara/  → S3 Event Notification → kpay-poc-tiara-queue → 클러스터 A
  ├── poc/batch/  → S3 Event Notification → kpay-poc-batch-queue → 클러스터 B
  └── cdc/        → S3 Event Notification → kpay-poc-cdc-queue   → 클러스터 C
```

> 그룹마다 SQS 큐 분리 → 각 클러스터가 자기 큐만 폴링 → 메시지 충돌 없음

---
## 시스템 정의

| 시스템 | 정의 |
|:---|:---|
| **티아라 (Tiara)** | 카카오페이 내부 행동 로그 시스템. 앱/서비스 내 사용자 행동(클릭, 결제 시도, 화면 이동 등)을 실시간 수집. Kafka → Iceberg(CEPH). PoC에서는 카카오페이가 Parquet으로 S3에 올려줌. |
| **Kudu** | Cloudera 전용 컬럼형 스토리지. PoC에서는 추출한 Parquet 파일을 S3에 올려줌. |
| **Iceberg** | 카카오페이 주력 테이블 포맷 (Starburst/Trino 호환). PoC에서는 Parquet으로 추출해서 S3에 올려줌. |

## 데이터 규모

| 그룹 | 테이블 | 컬럼 | 용량 | trigger |
|:---|:---|:---:|:---|:---|
| **tiara** | tiara_complete_log_raw | 75 | ~1.6TB (7일) / 84TB (1년) | processingTime |
| **tiara** | tiara_ns | - | - | processingTime |
| **batch** | payment_wide | 270 | 225GB/월 | availableNow |
| **batch** | an005d04 (Kudu) | - | - | availableNow |
| **batch** | bp501d01 (Kudu) | - | - | availableNow |
| **batch** | ETL 배치 45개 | - | 36TB | availableNow |

---
# [AWS 사전 설정] SQS + S3 Event Notification

Auto Loader `useNotifications=true` 동작에 필요한 AWS 리소스 설정.
**Databricks 코드 실행 전에 AWS 콘솔에서 먼저 완료해야 함.**

## Auto Loader 이벤트 흐름

```
카카오페이 → S3 파일 업로드
    └── S3 Event Notification 발생 (ObjectCreated)
          └── SQS Queue로 메시지 전달
                └── Auto Loader(cloudFiles)가 SQS 메시지 polling
                      └── 새 파일만 읽어 Delta에 증분 적재
                            └── 처리 완료 → Checkpoint에 기록 (중복 방지)
```

## SQS 큐 수량 — File Notification 모드 vs File Events 모드

| 모드 | 옵션 | SQS 큐 수 | 비고 |
|:---|:---|:---:|:---|
| File Notification 모드 | `useNotifications=true` | 스트림당 1개 (45테이블 = 45개) | 현재 방식 |
| **File Events 모드** | `useFileEvents=true` | **버킷당 1개** | Databricks 신규 권장 방식 |

> 45개 테이블이므로 **File Events 모드(큐 1개)** 사용 권장  
> 참고: [Databricks Auto Loader file detection modes](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/file-detection-modes)

---
## SQS 메시지 실제 예시

S3에 파일 업로드 시 SQS 큐에 들어오는 메시지 형태.  
AWS 콘솔 → SQS → 큐 선택 → "메시지 전송 및 수신" → "메시지 폴링"으로 직접 확인 가능.

```json
{
  "Type": "Notification",
  "MessageId": "f492db4d-fd21-59e1-9fd8-b93a7cf0e02d",
  "TopicArn": "arn:aws:sns:eu-north-1:302732540424:csms-topic-by-path-5db1f025...",
  "Subject": "Amazon S3 Notification",
  "Message": {
    "Records": [{
      "eventVersion": "2.1",
      "eventSource": "aws:s3",
      "awsRegion": "eu-north-1",
      "eventTime": "2026-05-08T06:15:44.886Z",
      "eventName": "ObjectCreated:Put",
      "userIdentity": {
        "principalId": "AWS:AIDAUM7CD7IEJ2IZI7Z7M"
      },
      "s3": {
        "bucket": {
          "name": "aws-s3-jimin-test",
          "arn": "arn:aws:s3:::aws-s3-jimin-test"
        },
        "object": {
          "key": "test/test10/xxxx.parquet",
          "size": 25395,
          "eTag": "123a1cdd2463070880f6918a1ac4a5e5"
        }
      }
    }]
  },
  "Timestamp": "2026-05-08T06:15:45.904Z"
}
```

**Auto Loader가 이 메시지에서 읽는 핵심 값:**

| 필드 | 값 | 용도 |
|:---|:---|:---|
| `eventName` | `ObjectCreated:Put` | 파일 생성 이벤트 확인 |
| `bucket.name` | `aws-s3-jimin-test` | 버킷 확인 |
| `object.key` | `test/raw_data/.../xxx.parquet` | 읽을 파일 경로 |
| `object.size` | `25395` | 파일 크기 (bytes) |

> `object.key`가 스트림의 `s3_path` 하위 경로인 경우에만 해당 스트림이 처리.  
> 다른 경로 메시지는 무시하고 큐에 남겨둠 → Visibility timeout 후 자동으로 다시 보임.

---
## Step 1. SQS 큐 생성

**AWS 콘솔 → SQS → 대기열 생성**

그룹별 3개 큐 + DLQ 1개, 총 4개 생성.

### 유형

| 유형 | 선택 여부 | 이유 |
|:---|:---:|:---|
| **Standard** | **선택** | Auto Loader는 순서 보장 불필요, 높은 처리량 필요 |
| FIFO | X | 초당 300건 제한 — Auto Loader에 부적합 |

### 큐 이름 (4개 생성)

```
kpay-poc-tiara-queue   ← tiara 그룹 전용 (클러스터 A가 폴링)
kpay-poc-batch-queue   ← batch 그룹 전용 (클러스터 B가 폴링)
kpay-poc-cdc-queue     ← CDC 그룹 전용  (클러스터 C가 폴링)
kpay-poc-dlq           ← 처리 실패 메시지 보관 (DLQ — 3개 큐 공용)
```

### 구성 (Configuration)  — 3개 큐 공통 적용

| 항목 | 권장값 | 이유 |
|:---|:---|:---|
| 가시성 제한 시간 (Visibility timeout) | **300초** | 기본 30초는 너무 짧음. Auto Loader 처리 중 중복 수신 방지 |
| 메시지 보존 기간 | 4일 (기본값) | 변경 불필요 |
| 배달 지연 | 0초 (기본값) | 파일 업로드 즉시 감지. 완성된 Parquet 파일 업로드이므로 지연 불필요 |
| 최대 메시지 크기 | 256KB (기본값) | 변경 불필요 |
| 메시지 수신 대기 시간 | **20초** | Long polling — 빈 큐 반복 조회 방지, API 비용 절감 |

### 암호화

```
서버 측 암호화 → 활성화
암호화 키 유형 → Amazon SQS 키(SSE-SQS)   ← 추가 비용 없음
```

> SSE-KMS는 Auto Loader polling 시마다 KMS API 호출 → 3개 큐 × 호출 빈도만큼 비용 누적. PoC에서는 SSE-SQS로 충분.

### 액세스 정책 구조 — S3 직접 전달 vs SNS 경유

실제 구현은 **SNS 경유(Fan-out) 방식**:

```
S3 업로드
  └── S3 Event Notification
        └── SNS Topic (csms-topic-by-path-...)   ← Fan-out 허브
              └── SQS Queue                       ← Auto Loader가 폴링
```

> SQS 메시지의 `TopicArn` 필드가 존재하는 것이 이 방식의 증거.  
> S3→SQS 직접 방식은 `TopicArn` 없이 `Records`만 존재.

### 액세스 정책 — SNS 경유 방식 (tiara 큐 예시)

**고급** 선택 후 아래 JSON 입력.  
SNS가 SQS로 메시지를 보낼 수 있도록 SNS Topic ARN을 Condition으로 허용.

```json
{
  "Version": "2012-10-17",
  "Id": "__default_policy_ID",
  "Statement": [
    {
      "Sid": "__owner_statement",
      "Effect": "Allow",
      "Principal": {
        "AWS": "arn:aws:iam::302732540424:root"
      },
      "Action": "SQS:*",
      "Resource": "arn:aws:sqs:eu-north-1:302732540424:kpay-poc-tiara-queue"
    },
    {
      "Sid": "allowSNSNotification",
      "Effect": "Allow",
      "Principal": {
        "AWS": "*"
      },
      "Action": "SQS:SendMessage",
      "Resource": "arn:aws:sqs:eu-north-1:302732540424:kpay-poc-tiara-queue",
      "Condition": {
        "ArnLike": {
          "aws:SourceArn": "arn:aws:sns:eu-north-1:302732540424:csms-topic-by-path-5db1f02514bd398e3d7c22e38297196b622be05e3ce549330c441a132e65fca2"
        }
      }
    }
  ]
}
```

> `Principal: "*"` + `Condition ArnLike SNS ARN` 조합 — SNS에서 오는 메시지만 허용하는 표준 패턴.  
> batch 큐 / cdc 큐도 동일 구조, `Resource` ARN과 필요 시 SNS ARN만 변경.

### 리드라이브 정책 (Dead Letter Queue)

처리 실패한 메시지를 버리지 않고 DLQ에 보관.

```
1. DLQ 큐 먼저 생성
   이름: kpay-poc-dlq
   유형: Standard

2. 각 원본 큐(tiara/batch/cdc) 리드라이브 정책 설정
   활성화: ON
   배달 못한 편지 대기열: kpay-poc-dlq
   최대 수신 횟수: 3
```

> DLQ에 메시지가 쌓이면 → Auto Loader가 특정 파일을 3번 읽으려다 실패한 것  
> 주요 원인: 파일 손상, 권한 오류, 스키마 불일치

### 태그

| Key | Value |
|:---|:---|
| `Project` | `kpay-poc` |
| `Environment` | `poc` |
| `Owner` | `jimin` |

---
## Step 2. S3 Event Notification 설정

**AWS 콘솔 → S3 → 버킷(aws-s3-jimin-test) → 속성(Properties) 탭 → 이벤트 알림 → 이벤트 알림 생성**

그룹별 SQS 큐와 대응하도록 이벤트 알림도 3개 생성.

### 주의: prefix 충돌

기존에 상위 prefix(`test/`)로 이벤트 알림이 있으면 하위 경로와 겹쳐서 아래 에러 발생:

```
Configuration is ambiguously defined. Cannot have overlapping suffixes
in two rules if the prefixes are overlapping for the same event type.
```

→ 기존 알림 삭제 후 새로 생성.

### S3 path 구조

```
버킷: aws-s3-jimin-test
  ├── poc/tiara/complete_log/collect_date=2025-09-01/xxxxxx.parquet
  ├── poc/batch/payment/collect_date=2025-09-01/xxxxxx.parquet
  └── cdc/mysql_table/20241020-000001.parquet

S3에 폴더는 없음 — "/" 는 key string의 일부
prefix 필터로 특정 경로 이벤트만 해당 SQS로 전달
```

### 이벤트 알림 3개 설정값

| 이름 | 접두사 (Prefix) | 접미사 (Suffix) | 이벤트 유형 | 대상 SQS ARN |
|:---|:---|:---|:---|:---|
| kpay-tiara-event | `poc/tiara/` | `.parquet` | `s3:ObjectCreated:*` | `arn:aws:sqs:eu-north-1:302732540424:kpay-poc-tiara-queue` |
| kpay-batch-event | `poc/batch/` | `.parquet` | `s3:ObjectCreated:*` | `arn:aws:sqs:eu-north-1:302732540424:kpay-poc-batch-queue` |
| kpay-cdc-event   | `cdc/`       | `.parquet` | `s3:ObjectCreated:*` | `arn:aws:sqs:eu-north-1:302732540424:kpay-poc-cdc-queue` |

> SQS ARN 확인: SQS 콘솔 → 큐 클릭 → 상단 ARN 복사  
> 접미사 `.parquet`: `_SUCCESS` 같은 메타파일 무시  
> prefix가 겹치지 않으므로(`poc/tiara/`, `poc/batch/`, `cdc/`) 충돌 없음

### IAM Role 필요 권한

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetBucketNotification",
        "s3:PutBucketNotification"
      ],
      "Resource": "arn:aws:s3:::aws-s3-jimin-test"
    },
    {
      "Effect": "Allow",
      "Action": [
        "sqs:CreateQueue", "sqs:DeleteQueue",
        "sqs:GetQueueAttributes", "sqs:GetQueueUrl",
        "sqs:ReceiveMessage", "sqs:SendMessage",
        "sqs:SetQueueAttributes", "sqs:DeleteMessage"
      ],
      "Resource": "arn:aws:sqs:eu-north-1:302732540424:kpay-poc-*"
    }
  ]
}
```

---
# 0. 파라미터 설정

> **보안 원칙**: 버킷명·IAM ARN 등 민감값은 노트북 셀에 직접 입력하지 않음.  
> `dbutils.secrets`로 Secret Scope에서 참조.

In [ ]:
dbutils.widgets.text("group", "tiara", "실행 그룹 (tiara | batch | cdc)")

GROUP  = dbutils.widgets.get("group")
REGION = "eu-north-1"

# 민감값은 Secret Scope에서 참조 (노트북에 직접 입력 금지)
BUCKET     = dbutils.secrets.get(scope="kpay-poc", key="s3-bucket-name")
ACCOUNT_ID = dbutils.secrets.get(scope="kpay-poc", key="aws-account-id")
IAM_ROLE   = dbutils.secrets.get(scope="kpay-poc", key="iam-role-arn")

CHECKPOINT_BASE = f"s3://{BUCKET}/poc/checkpoints"

# 그룹별 SQS 큐 분리 — 각 클러스터가 자기 큐만 폴링
SQS_MAP = {
    "tiara": f"arn:aws:sqs:{REGION}:{ACCOUNT_ID}:kpay-poc-tiara-queue",
    "batch": f"arn:aws:sqs:{REGION}:{ACCOUNT_ID}:kpay-poc-batch-queue",
    "cdc"  : f"arn:aws:sqs:{REGION}:{ACCOUNT_ID}:kpay-poc-cdc-queue",
}

if GROUP not in SQS_MAP:
    raise ValueError(f"알 수 없는 group 값: {GROUP}. 'tiara' | 'batch' | 'cdc' 만 허용")

SQS_ARN = SQS_MAP[GROUP]

print(f"실행 그룹  : {GROUP}")
print(f"S3 버킷    : {BUCKET}")
print(f"SQS ARN    : {SQS_ARN}")
print(f"체크포인트 : {CHECKPOINT_BASE}")

---
# 1. Storage Credential + External Location

In [ ]:
spark.sql(f"""
CREATE STORAGE CREDENTIAL IF NOT EXISTS kpay_poc_credential
  WITH IAM_ROLE (CREDENTIAL = '{IAM_ROLE}');
""")

spark.sql(f"""
CREATE EXTERNAL LOCATION IF NOT EXISTS kpay_poc_location
  URL 's3://{BUCKET}/test/raw_data/'
  WITH (STORAGE CREDENTIAL kpay_poc_credential);
""")

spark.sql(f"LIST 's3://{BUCKET}/test/raw_data/';").show(5, truncate=False)

---
# 2. 카탈로그 / 스키마 생성

In [ ]:
spark.sql("CREATE CATALOG IF NOT EXISTS kpay_poc;")
spark.sql("CREATE SCHEMA IF NOT EXISTS kpay_poc.bronze;")
print("카탈로그 / 스키마 준비 완료")

---
# 3. Checkpoint 구조

Auto Loader는 테이블마다 2개의 체크포인트 경로를 사용한다.

```
s3://<BUCKET>/poc/checkpoints/
  ├── tiara_complete_log_raw/
  │     ├── schema/          ← cloudFiles.schemaLocation  (추론된 스키마 캐싱)
  │     └── checkpoint/      ← checkpointLocation
  │           ├── commits/   ← 완료된 micro-batch 번호
  │           ├── offsets/   ← 배치별 처리 파일 범위
  │           └── sources/   ← 처리 완료된 S3 파일 목록 (중복 방지 핵심)
  └── ... (테이블 수만큼)
```

| 경로 | 역할 | 없으면? |
|:---|:---|:---|
| `schema/` | 추론된 스키마 캐싱 | 처음 실행 시 자동 생성 |
| `checkpoint/commits/` | 완료된 배치 번호 기록 | 재기동 시 처음부터 재처리 |
| `checkpoint/offsets/` | 배치별 처리 파일 범위 | 중복 적재 발생 가능 |
| `checkpoint/sources/` | 처리 완료된 S3 파일 목록 | 동일 파일 재처리 발생 |

In [ ]:
def check_checkpoint_status(table_id):
    schema_path = f"{CHECKPOINT_BASE}/{table_id}/schema"
    ckpt_path   = f"{CHECKPOINT_BASE}/{table_id}/checkpoint"

    def path_exists(path):
        try:
            dbutils.fs.ls(path)
            return True
        except:
            return False

    schema_ok = path_exists(schema_path)
    ckpt_ok   = path_exists(ckpt_path)
    status    = "재개 가능" if (schema_ok and ckpt_ok) else "신규 시작"
    print(f"  [{table_id}] schema={'존재' if schema_ok else '없음'} / checkpoint={'존재' if ckpt_ok else '없음'} → {status}")


def reset_checkpoint(table_id, confirm=False):
    """체크포인트 초기화. confirm=True 명시 필요."""
    if not confirm:
        print(f"[주의] reset_checkpoint('{table_id}', confirm=True) 로 실행해야 삭제됩니다.")
        return
    path = f"{CHECKPOINT_BASE}/{table_id}"
    dbutils.fs.rm(path, recurse=True)
    print(f"[삭제 완료] {path} → 다음 실행 시 처음부터 재적재됩니다.")


print("체크포인트 유틸 로드 완료")

---
# 4. 테이블 설정 정의

SQS 큐 1개(`kpay-poc-raw-data`)를 전 테이블이 공유.  
각 스트림은 `s3_path`로 자기 경로에 해당하는 파일만 처리.

In [ ]:
TIARA_TABLES = [
    {
        "target"    : "kpay_poc.bronze.tiara_complete_log_raw",
        "s3_path"   : f"s3://{BUCKET}/poc/tiara/complete_log/",
        "trigger"   : {"processingTime": "10 minutes"},
        "cluster_by": None,
    },
    {
        "target"    : "kpay_poc.bronze.tiara_ns",
        "s3_path"   : f"s3://{BUCKET}/poc/tiara/ns/",
        "trigger"   : {"processingTime": "10 minutes"},
        "cluster_by": None,
    },
]

BATCH_TABLES = [
    {
        "target"    : "kpay_poc.bronze.payment_wide",
        "s3_path"   : f"s3://{BUCKET}/poc/batch/payment/",
        "trigger"   : {"availableNow": True},
        "cluster_by": ["payment_id", "created_date"],   # Liquid Clustering
    },
    {
        "target"    : "kpay_poc.bronze.an005d04",
        "s3_path"   : f"s3://{BUCKET}/poc/batch/kudu/an005d04/",
        "trigger"   : {"availableNow": True},
        "cluster_by": None,
    },
    {
        "target"    : "kpay_poc.bronze.bp501d01",
        "s3_path"   : f"s3://{BUCKET}/poc/batch/kudu/bp501d01/",
        "trigger"   : {"availableNow": True},
        "cluster_by": None,
    },
    # ETL 배치 45개 — 실제 테이블명 확정 후 동일 패턴으로 추가
    # {"target": "kpay_poc.bronze.etl_xxx",
    #  "s3_path": f"s3://{BUCKET}/poc/batch/etl/xxx/",
    #  "trigger": {"availableNow": True}, "cluster_by": None},
]

CDC_TABLES = [
    # MySQL CDC — DMS가 S3에 쓴 변경분 파일 (Op=I/U/D 포함)
    # 실제 테이블명 확정 후 추가
    # {"target": "kpay_poc.bronze.mysql_xxx_cdc",
    #  "s3_path": f"s3://{BUCKET}/cdc/xxx/",
    #  "trigger": {"processingTime": "5 minutes"}, "cluster_by": None},
]

TABLE_MAP = {
    "tiara": TIARA_TABLES,
    "batch": BATCH_TABLES,
    "cdc"  : CDC_TABLES,
}

TABLES = TABLE_MAP[GROUP]

print(f"[{GROUP}] 처리 대상 테이블 {len(TABLES)}개  /  SQS: {SQS_ARN}")
for t in TABLES:
    print(f"  - {t['target']:<50} {t['s3_path']}")

In [ ]:
# 실행 전 체크포인트 상태 확인
print(f"=== [{GROUP}] 체크포인트 상태 ===")
for t in TABLES:
    check_checkpoint_status(t["target"].split(".")[-1])

---
# 5. Auto Loader 실행

### `_rescued_data` 컬럼

카카오페이가 Iceberg/Kudu에서 Parquet으로 추출하는 과정에서 타입 미스매치 발생 가능.  
`cloudFiles.rescuedDataColumn` 옵션으로 스키마와 맞지 않는 행을 JSON으로 별도 저장.

```
정상 행     → 각 컬럼에 정상 적재
미스매치 행 → _rescued_data 컬럼에 JSON으로 저장 (데이터 유실 없음)
```

### `_metadata` 컬럼 (Auto Loader 파일 추적)

Auto Loader가 읽은 파일 정보를 Bronze 테이블에 함께 저장.  
파일 단위 트러블슈팅, 재처리 범위 특정, 어떤 파일에서 왔는지 추적에 활용.

| 컬럼명 | 원본 | 설명 |
|:---|:---|:---|
| `_source_file_path` | `_metadata.file_path` | 파일 전체 경로 (`s3://bucket/path/file.parquet`) |
| `_source_file_name` | `_metadata.file_name` | 파일명만 (`file.parquet`) |
| `_source_file_size` | `_metadata.file_size` | 파일 크기 (bytes) |
| `_source_modified_at` | `_metadata.file_modification_time` | 파일 최종 수정 시각 |

> `_metadata`는 Auto Loader(cloudFiles)가 자동으로 제공하는 숨김 컬럼.  
> `.withColumn()`으로 명시적으로 추출해야 Delta 테이블에 영구 저장됨.

In [ ]:
from pyspark.sql import functions as F


def create_table_if_needed(target, cluster_by):
    if cluster_by is None:
        return
    cols = ", ".join(cluster_by)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {target}
        CLUSTER BY ({cols})
        TBLPROPERTIES ('delta.enableDeletionVectors' = 'true')
    """)
    print(f"  Liquid Clustering 테이블 생성: CLUSTER BY ({cols})")


def run_auto_loader(table_config):
    target   = table_config["target"]
    s3_path  = table_config["s3_path"]
    trigger  = table_config["trigger"]
    table_id = target.split(".")[-1]

    schema_ckpt = f"{CHECKPOINT_BASE}/{table_id}/schema"
    data_ckpt   = f"{CHECKPOINT_BASE}/{table_id}/checkpoint"

    print(f"\n[시작] {target}")
    print(f"  소스       : {s3_path}")
    print(f"  SQS        : {SQS_ARN}")
    print(f"  schema     : {schema_ckpt}")
    print(f"  checkpoint : {data_ckpt}")
    print(f"  trigger    : {trigger}")

    create_table_if_needed(target, table_config["cluster_by"])

    query = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "parquet")
            .option("cloudFiles.useNotifications", "true")              # S3 Event Notifications + SQS
            .option("cloudFiles.sqsArn", SQS_ARN)                      # 그룹별 SQS 큐
            .option("cloudFiles.inferColumnTypes", "true")              # 타입 자동 추론
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # 신규 컬럼 자동 추가
            .option("cloudFiles.schemaLocation", schema_ckpt)           # 스키마 캐싱
            .option("cloudFiles.rescuedDataColumn", "_rescued_data")    # 타입 미스매치 안전망
            .load(s3_path)
            # _metadata: Auto Loader가 제공하는 파일 정보 (파일 추적 · 트러블슈팅용)
            .withColumn("_source_file_path",    F.col("_metadata.file_path"))
            .withColumn("_source_file_name",    F.col("_metadata.file_name"))
            .withColumn("_source_file_size",    F.col("_metadata.file_size"))
            .withColumn("_source_modified_at",  F.col("_metadata.file_modification_time"))
        .writeStream
            .format("delta")
            .option("checkpointLocation", data_ckpt)   # 처리 완료 파일 기록 → 중복 방지
            .option("mergeSchema", "true")
            .trigger(**trigger)
            .toTable(target)
    )
    return query


print(f"=== [{GROUP}] Auto Loader 적재 시작 ===")
queries = []
for table in TABLES:
    q = run_auto_loader(table)
    queries.append(q)

print(f"\n총 {len(queries)}개 스트림 / SQS: {SQS_ARN}")

In [ ]:
for q in queries:
    q.awaitTermination()

print(f"\n=== [{GROUP}] 전체 적재 완료 ===")

---
# 6. 적재 결과 확인

In [ ]:
print(f"{'테이블':<50} {'행 수':>15} {'rescued 행':>12} {'trigger'}")
print("-" * 95)
for t in TABLES:
    target  = t["target"]
    trigger = list(t["trigger"].keys())[0]
    try:
        cnt     = spark.sql(f"SELECT COUNT(*) as c FROM {target}").collect()[0]["c"]
        rescued = spark.sql(f"SELECT COUNT(*) as c FROM {target} WHERE _rescued_data IS NOT NULL").collect()[0]["c"]
        print(f"{target:<50} {cnt:>15,} {rescued:>12,} {trigger}")
        if rescued > 0:
            print(f"  [주의] {rescued:,}개 행이 _rescued_data에 저장됨 → 타입 미스매치 확인 필요")
    except Exception as e:
        print(f"{target:<50} {'오류':>15} {e}")

In [ ]:
# rescued_data 샘플 확인 (미스매치 원인 파악용)
target = TABLES[0]["target"]
spark.sql(f"""
    SELECT _rescued_data
    FROM {target}
    WHERE _rescued_data IS NOT NULL
    LIMIT 5
""").show(truncate=False)

In [ ]:
# 체크포인트 최종 상태
print(f"=== [{GROUP}] 체크포인트 최종 상태 ===")
for t in TABLES:
    check_checkpoint_status(t["target"].split(".")[-1])

---
# 7. 체크포인트 초기화 (필요 시만)

> 스키마 대폭 변경, 소스 경로 변경, 재적재 필요 시 사용.  
> `confirm=True` 명시 필요.

In [ ]:
# reset_checkpoint("tiara_complete_log_raw", confirm=True)

---
# 8. CDC 적재 — DMS + Auto Loader + APPLY CHANGES INTO

## 흐름

```
MySQL (TPS 7,040)
  └── AWS DMS (CDC 캡처 — 트랜잭션 로그 읽기)
        └── S3에 변경분 파일 저장 (Op=I/U/D 포함 Parquet)
              └── Auto Loader → Bronze (CDC raw, append only)
                    └── APPLY CHANGES INTO → Silver (최신 상태 유지)
```

## DMS가 S3에 쓰는 파일 구조

```
s3://bucket/cdc/mysql_table/
  ├── LOAD00001.parquet        ← 초기 전체 적재 (Op=I)
  ├── 20241020-000001.parquet  ← 변경분 (Op=I/U/D 혼합)
  └── 20241020-000002.parquet
```

| `Op` 값 | 의미 |
|:---:|:---|
| `I` | INSERT |
| `U` | UPDATE |
| `D` | DELETE |

## APPLY CHANGES INTO vs MERGE INTO

| | `MERGE INTO` | `APPLY CHANGES INTO` |
|:---|:---|:---|
| 사용 위치 | 일반 Spark SQL | SDP(DLT) 파이프라인 전용 |
| CDC 처리 | Op 컬럼 직접 핸들링 필요 | **자동 처리** |
| 순서 보장 | 직접 정렬 필요 | `sequence_by`로 자동 보장 |
| 늦게 도착한 데이터 | 직접 처리 필요 | **자동 처리** |

In [ ]:
import dlt
from pyspark.sql.functions import expr

# ---------------------------------------------------------------------------
# Bronze — DMS CDC 파일 Auto Loader로 적재
# ---------------------------------------------------------------------------
@dlt.table(
    name="bronze_mysql_cdc",
    comment="MySQL CDC raw — DMS가 S3에 쓴 변경분 파일 (Op=I/U/D 포함)"
)
def bronze_mysql_cdc():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "parquet")
            .option("cloudFiles.useNotifications", "true")
            .option("cloudFiles.sqsArn", SQS_ARN)
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaLocation", f"{CHECKPOINT_BASE}/mysql_cdc/schema")
            .option("cloudFiles.rescuedDataColumn", "_rescued_data")
            .load(f"s3://{BUCKET}/cdc/mysql_table/")
    )


# ---------------------------------------------------------------------------
# Silver — APPLY CHANGES INTO로 CDC 자동 처리 (최신 상태만 유지)
# ---------------------------------------------------------------------------
dlt.apply_changes(
    target      = "silver_mysql",           # 최신 상태를 유지할 Silver 테이블
    source      = "bronze_mysql_cdc",       # CDC raw Bronze
    keys        = ["id"],                   # PK — 실제 PK 컬럼명으로 변경
    sequence_by = "commit_timestamp",       # 변경 순서 기준 컬럼 (DMS 제공)
    apply_as_deletes = expr("Op = 'D'"),    # D → DELETE 처리
    except_column_list = ["Op"]             # Op 컬럼은 Silver에서 제외
)